### Supplement 1: Basic chat completion (`1-basic.py`)

Send two messages to the model and print the text it writes back.

**The key point:** `client.chat.completions.create(...)` sends your `messages` to OpenAI as JSON. The SDK turns the JSON reply into a `ChatCompletion` object, and the text you want is at `completion.choices[0].message.content`.

```
messages            a list of dicts, written by you
   │  client.chat.completions.create(model=..., messages=messages)
   ▼
OpenAI's server     runs the model and replies with JSON text
   │  the SDK turns that JSON into an object
   ▼
completion          a ChatCompletion object
   └─ .choices[0].message.content    a plain str: the limerick
```

#### 1. Imports and the API key
`load_dotenv()` looks for a `.env` file, starting in this folder and moving up, and copies its variables (including `OPENAI_API_KEY`) into `os.environ`. `OpenAI()` reads the key from there in the next step.

- **The order matters:** `load_dotenv()` has to run before `OpenAI()`, or the key isn't in `os.environ` yet when the client looks for it.
- **A missing key fails early, a wrong key fails late:** With no key, `OpenAI()` raises `OpenAIError: Missing credentials` while building the client. A wrong key builds a client without complaint, and only the first request is refused.
- **The long form builds the same client:** `OpenAI(api_key=os.getenv("OPENAI_API_KEY"))`, used in Supplement 2, reads the same `os.environ` itself and hands the key over. Write `api_key=` only when the key comes from somewhere else, such as a variable with another name.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

import json  # added for the prints below

load_dotenv()

#### 2. The client
`OpenAI` is a class, a blueprint. `client` is the object built from it: It holds your API key and the server address, and every request goes through it.

In [ ]:
client = OpenAI()

print("type(client):", type(client))
print("client.base_url:", client.base_url)

type(client): <class 'openai.OpenAI'>
client.base_url: https://api.openai.com/v1/


#### 3. The messages
A plain Python list of dicts; no SDK class is involved. Each dict has a `role` and some `content`:
- `system`: instructions for how the model should behave.
- `user`: the request itself.

`1-basic.py` writes this list inside the `create()` call. Here it gets its own variable so it can be printed.

**Two more roles you'll meet:** `assistant` marks the model's own replies, and `tool` carries a tool's result back to the model (Supplement 3).

**The API remembers nothing between calls.** To continue a chat, you send the whole conversation again, with the model's earlier reply tagged `assistant`:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
messages for the next call
   system      You're a helpful assistant.
   user        Write a limerick about the Python programming language.
   assistant   There once was a language named Python, ...
   user        Explain the humor in the last line.
</pre>

Tagged `system`, the old reply would read as new instructions instead of the model's own words.

In [3]:
messages = [
    {"role": "system", "content": "You're a helpful assistant."},
    {
        "role": "user",
        "content": "Write a limerick about the Python programming language.",
    },
]

print("type(messages):", type(messages))
print("len(messages):", len(messages))
print("type(messages[0]):", type(messages[0]))
print()
print(json.dumps(messages, indent=2))  # the same list as JSON text, the form it travels in

type(messages): <class 'list'>
len(messages): 2
type(messages[0]): <class 'dict'>

[
  {
    "role": "system",
    "content": "You're a helpful assistant."
  },
  {
    "role": "user",
    "content": "Write a limerick about the Python programming language."
  }
]


#### 4. The API call
`create()` sends `model` and `messages` to OpenAI and waits for the reply. It returns a `ChatCompletion` object, not a string. `gpt-5-nano` is a reasoning model: it thinks privately before it writes, so the call can take a while.

**Only text crosses the network.** Your list goes out as JSON text, the reply comes back as JSON text, and the SDK turns that text into the `ChatCompletion` object on your machine. Nothing checks `messages` on the way out: A typo like `"rol"` is rejected by OpenAI's server with a 400 error, not by the SDK.

In [4]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
)

In [5]:
print(type(completion))
# model_dump_json() is a Pydantic method: it writes the whole object out as JSON text.
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EUYdtdCsEb9BRbJKe45qMMbhWkc0V",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "There once was a language named Python,\nIts syntax was friendly and bright, Python,\nIndents guided every careful line,\nLibraries and modules kept it all in line,\nAnd coders cheered, praising Python.",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1790951713,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 3828,
    "prompt_tokens": 26,
    "total_tokens": 3854,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "a

#### 5. Getting the text out
The text is four levels down, and each level is a different SDK class until `.content`, which is a plain `str`. `type(x)` prints the class of `x`; the next cell prints it for each level.
- `choices` is a list because you can ask for several alternative replies (the `n` parameter). By default you get one, at index `0`.
- `finish_reason` is `'stop'` when the model ended the reply by itself.
- `model_dump()` goes the other way: It turns the object back into a plain dict, and `model_dump_json()`, used above, turns it into JSON text. Supplement 2 lists all of these conversions.

**Where `model_dump()` comes from: The MRO.** MRO is short for **Method Resolution Order**: The list of classes Python searches, in order, when you call a method on an object. A class that inherits from another gets all of its methods, and `ChatCompletion` inherits from Pydantic, two levels up:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
ChatCompletion        OpenAI's class for the reply (id, choices ...)
   │  inherits from
   ▼
openai.BaseModel      OpenAI's wrapper (to_dict, to_json ...)
   │  inherits from
   ▼
pydantic.BaseModel    Pydantic's class (model_dump, model_dump_json)
   │  inherits from
   ▼
object                the base of every Python class
</pre>

For `completion.model_dump()`, Python looks in `ChatCompletion` first and finds only fields there, then in `openai.BaseModel`, and finds `model_dump` in `pydantic.BaseModel`, where it stops. That's why the comment in step 4 calls `model_dump_json()` a Pydantic method. `print(type(completion).mro())` prints the list for you.

In [6]:
print(type(completion.choices))
print(type(completion.choices[0]))
print(type(completion.choices[0].message))
print(type(completion.choices[0].message.content))
print()
print("finish_reason:", completion.choices[0].finish_reason)
print("role:", completion.choices[0].message.role)

<class 'list'>
<class 'openai.types.chat.chat_completion.Choice'>
<class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
<class 'str'>

finish_reason: stop
role: assistant


In [7]:
response = completion.choices[0].message.content
print(response)

There once was a language named Python,
Its syntax was friendly and bright, Python,
Indents guided every careful line,
Libraries and modules kept it all in line,
And coders cheered, praising Python.


#### 6. Token usage
You pay per token (a chunk of text, roughly ¾ of an English word) for what the model reads (`prompt_tokens`) and what it writes (`completion_tokens`). `completion_tokens` includes the hidden reasoning tokens, which is why it is far larger than a five-line limerick needs. So you can't tell the cost from the length of the reply.

In [8]:
usage = completion.usage

print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("reasoning_tokens:", usage.completion_tokens_details.reasoning_tokens)
print("total_tokens:", usage.total_tokens)

prompt_tokens: 26
completion_tokens: 3828
reasoning_tokens: 3776
total_tokens: 3854
